In [1]:
from collections import namedtuple
import math

# Named tuples for agent convenience.
# Planets and fleets share a common [id, owner, x, y, ...] prefix.
Planet = namedtuple(
    "Planet", ["id", "owner", "x", "y", "radius", "ships", "production"]
)
Fleet = namedtuple(
    "Fleet", ["id", "owner", "x", "y", "angle", "from_planet_id", "ships"]
)

# Constants
BOARD_SIZE = 100.0
CENTER = BOARD_SIZE / 2.0
SUN_RADIUS = 10.0
ROTATION_RADIUS_LIMIT = 50.0
COMET_RADIUS = 1.0
COMET_PRODUCTION = 1
PLANET_CLEARANCE = 7
MIN_PLANET_GROUPS = 5
MAX_PLANET_GROUPS = 10
MIN_STATIC_GROUPS = 3
COMET_SPAWN_STEPS = [50, 150, 250, 350, 450]

CENTER_X = 50.0
CENTER_Y = 50.0
MAX_SPEED = 6.0
MAX_NB_STEP = 500


def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def point_to_segment_distance(p, v, w):
    """Minimum distance from point p to line segment v-w."""
    l2 = (v[0] - w[0]) ** 2 + (v[1] - w[1]) ** 2
    if l2 == 0.0:
        return distance(p, v)
    t = max(
        0, min(1, ((p[0] - v[0]) * (w[0] - v[0]) + (p[1] - v[1]) * (w[1] - v[1])) / l2)
    )
    projection = (v[0] + t * (w[0] - v[0]), v[1] + t * (w[1] - v[1]))
    return distance(p, projection)

def interpreter(obs, actions, step, num_agents=2):
    # configuration = env.configuration
    obs0 = obs

    # Remove expired comets before fleet launch so agents can't act on them
    expired_comet_pids = []
    for group in obs0.comets:
        idx = group["path_index"]
        for i, pid in enumerate(group["planet_ids"]):
            if idx >= len(group["paths"][i]):
                expired_comet_pids.append(pid)
    if expired_comet_pids:
        expired_set = set(expired_comet_pids)
        obs0.planets = [p for p in obs0.planets if p[0] not in expired_set]
        obs0.initial_planets = [
            p for p in obs0.initial_planets if p[0] not in expired_set
        ]
        obs0.comet_planet_ids = [
            pid for pid in obs0.comet_planet_ids if pid not in expired_set
        ]
        for group in obs0.comets:
            group["planet_ids"] = [
                pid for pid in group["planet_ids"] if pid not in expired_set
            ]
        obs0.comets = [g for g in obs0.comets if g["planet_ids"]]

    # Spawn extra-solar comets at designated steps
    # step = get(obs0, "step", 0)
    # comet_speed = configuration.cometSpeed
    # if (step + 1) in COMET_SPAWN_STEPS:
    #     # Derive a per-spawn RNG from the episode seed so comet shape and
    #     # ship counts are reproducible. Seed lives on env.info to keep it
    #     # hidden from agents (see init block above).
    #     env_info = getattr(env, "info", None) or {}
    #     episode_seed = env_info.get("seed", 0) or 0
    #     comet_rng = random.Random(f"orbit_wars-comet-{episode_seed}-{step + 1}")
    #     comet_paths = generate_comet_paths(
    #         obs0.initial_planets,
    #         obs0.angular_velocity,
    #         step + 1,
    #         obs0.comet_planet_ids,
    #         comet_speed,
    #         rng=comet_rng,
    #     )
    #     if comet_paths:
    #         next_id = max(p[0] for p in obs0.planets) + 1
    #         comet_ships = min(
    #             comet_rng.randint(1, 99),
    #             comet_rng.randint(1, 99),
    #             comet_rng.randint(1, 99),
    #             comet_rng.randint(1, 99),
    #         )
    #         group = {"planet_ids": [], "paths": comet_paths, "path_index": -1}
    #         for i, p_path in enumerate(comet_paths):
    #             pid = next_id + i
    #             group["planet_ids"].append(pid)
    #             obs0.comet_planet_ids.append(pid)
    #             # Start off-board; first advancement will place at path[0]
    #             planet = [
    #                 pid,
    #                 -1,
    #                 -99,
    #                 -99,
    #                 COMET_RADIUS,
    #                 comet_ships,
    #                 COMET_PRODUCTION,
    #             ]
    #             obs0.planets.append(planet)
    #             obs0.initial_planets.append(planet[:])
    #         obs0.comets.append(group)

    # 0. Fleet Launch
    def process_moves(player_id, action):
        if not action or not isinstance(action, list):
            return
        for move in action:
            if len(move) != 3:
                continue
            from_id, angle, ships = move
            ships = int(ships)  # Sanitize to integer

            from_planet = next((p for p in obs0.planets if p[0] == from_id), None)

            if from_planet and from_planet[1] == player_id:
                if from_planet[5] >= ships and ships > 0:
                    from_planet[5] -= ships
                    # Start fleet just outside the planet so it doesn't
                    # immediately collide with its origin.
                    start_x = from_planet[2] + math.cos(angle) * (from_planet[4] + 0.1)
                    start_y = from_planet[3] + math.sin(angle) * (from_planet[4] + 0.1)
                    obs0.fleets.append(
                        [
                            obs0.next_fleet_id,
                            player_id,
                            start_x,
                            start_y,
                            angle,
                            from_id,
                            ships,
                        ]
                    )
                    obs0.next_fleet_id += 1

    for i in range(num_agents):
        process_moves(i, actions[i])

    # 1. Production
    for planet in obs0.planets:
        if planet[1] != -1:
            planet[5] += planet[6]

    # 2. Fleet Movement (with continuous collision detection)
    # Speed scales with fleet size: 1 ship = 1/turn, max = shipSpeed (default 6)
    max_speed = MAX_SPEED
    fleets_to_remove = []
    combat_lists = {p[0]: [] for p in obs0.planets}

    for fleet in obs0.fleets:
        angle = fleet[4]
        ships = fleet[6]
        speed = 1.0 + (max_speed - 1.0) * (math.log(ships) / math.log(1000)) ** 1.5
        speed = min(speed, max_speed)
        old_pos = (fleet[2], fleet[3])
        fleet[2] += math.cos(angle) * speed
        fleet[3] += math.sin(angle) * speed
        new_pos = (fleet[2], fleet[3])

        # Check if fleet path intersected any planet (continuous collision).
        # Check planets first so fast fleets that would overshoot the bounds
        # or sun still get credit for hitting a planet along the way.
        hit_planet = False
        for planet in obs0.planets:
            planet_pos = (planet[2], planet[3])
            if point_to_segment_distance(planet_pos, old_pos, new_pos) < planet[4]:
                combat_lists[planet[0]].append(fleet)
                fleets_to_remove.append(fleet)
                hit_planet = True
                break
        if hit_planet:
            continue

        # Check if fleet went out of bounds
        if not (0 <= fleet[2] <= BOARD_SIZE and 0 <= fleet[3] <= BOARD_SIZE):
            fleets_to_remove.append(fleet)
            continue

        # Check if fleet path crossed the sun
        if point_to_segment_distance((CENTER, CENTER), old_pos, new_pos) < SUN_RADIUS:
            fleets_to_remove.append(fleet)
            continue

    # 3. Planet Movement & Sweep
    angular_velocity = obs0.angular_velocity
    comet_pid_set = set(obs0.comet_planet_ids)
    initial_by_id = {p[0]: p for p in obs0.initial_planets}

    def sweep_fleets(planet, old_pos, new_pos):
        """Check if any fleet is caught by a planet moving from old to new."""
        if old_pos == new_pos:
            return
        for fleet in obs0.fleets:
            if fleet not in fleets_to_remove:
                if (
                    point_to_segment_distance((fleet[2], fleet[3]), old_pos, new_pos)
                    < planet[4]
                ):
                    combat_lists[planet[0]].append(fleet)
                    fleets_to_remove.append(fleet)

    # Regular planet rotation
    for planet in obs0.planets:
        if planet[0] in comet_pid_set:
            continue
        initial_p = initial_by_id.get(planet[0])
        if not initial_p:
            continue
        dx = initial_p[2] - CENTER
        dy = initial_p[3] - CENTER
        r = math.sqrt(dx**2 + dy**2)
        old_pos = (planet[2], planet[3])

        if r + planet[4] < ROTATION_RADIUS_LIMIT:
            initial_angle = math.atan2(dy, dx)
            current_angle = initial_angle + angular_velocity * step
            planet[2] = CENTER + r * math.cos(current_angle)
            planet[3] = CENTER + r * math.sin(current_angle)

        sweep_fleets(planet, old_pos, (planet[2], planet[3]))

    # Comet movement along pre-computed paths
    expired_comet_pids = []
    for group in obs0.comets:
        group["path_index"] += 1
        idx = group["path_index"]
        for i, pid in enumerate(group["planet_ids"]):
            planet = next((p for p in obs0.planets if p[0] == pid), None)
            if planet is None:
                continue
            p_path = group["paths"][i]
            if idx >= len(p_path):
                expired_comet_pids.append(pid)
            else:
                old_pos = (planet[2], planet[3])
                planet[2] = p_path[idx][0]
                planet[3] = p_path[idx][1]
                # Skip sweep on first placement (old_pos is off-board placeholder)
                if old_pos[0] >= 0:
                    sweep_fleets(planet, old_pos, (planet[2], planet[3]))

    # Remove expired comets immediately
    if expired_comet_pids:
        expired_set = set(expired_comet_pids)
        obs0.planets = [p for p in obs0.planets if p[0] not in expired_set]
        obs0.initial_planets = [
            p for p in obs0.initial_planets if p[0] not in expired_set
        ]
        obs0.comet_planet_ids = [
            pid for pid in obs0.comet_planet_ids if pid not in expired_set
        ]
        for group in obs0.comets:
            group["planet_ids"] = [
                pid for pid in group["planet_ids"] if pid not in expired_set
            ]
        obs0.comets = [g for g in obs0.comets if g["planet_ids"]]

    obs0.fleets = [f for f in obs0.fleets if f not in fleets_to_remove]

    # 4. Combat Resolution
    for pid, planet_fleets in combat_lists.items():
        planet = next((p for p in obs0.planets if p[0] == pid), None)
        if not planet or not planet_fleets:
            continue

        # Sum ships per player
        player_ships = {}
        for fleet in planet_fleets:
            owner = fleet[1]
            player_ships[owner] = player_ships.get(owner, 0) + fleet[6]

        if not player_ships:
            continue

        sorted_players = sorted(
            player_ships.items(), key=lambda item: item[1], reverse=True
        )
        top_player, top_ships = sorted_players[0]

        if len(sorted_players) > 1:
            second_ships = sorted_players[1][1]
            survivor_ships = top_ships - second_ships

            if sorted_players[0][1] == sorted_players[1][1]:
                survivor_ships = 0

            survivor_owner = top_player if survivor_ships > 0 else -1
        else:
            survivor_owner = top_player
            survivor_ships = top_ships

        if survivor_ships > 0:
            if planet[1] == survivor_owner:
                planet[5] += survivor_ships
            else:
                planet[5] -= survivor_ships
                if planet[5] < 0:
                    planet[1] = survivor_owner
                    planet[5] = abs(planet[5])

    obs1 = {}
    obs1["planets"] = obs0.planets
    obs1["initial_planets"] = obs0.initial_planets
    obs1["fleets"] = obs0.fleets
    obs1["next_fleet_id"] = obs0.next_fleet_id
    obs1["comets"] = obs0.comets
    obs1["comet_planet_ids"] = obs0.comet_planet_ids

    terminated = False
    if step >= MAX_NB_STEP - 2:
        terminated = True

    alive_players = set()
    for p in obs0.planets:
        if p[1] != -1:
            alive_players.add(p[1])
    for f in obs0.fleets:
        alive_players.add(f[1])

    if len(alive_players) <= 1:
        terminated = True


    return obs1

# Test cases

In [2]:
import copy, math
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# ---------------------------------------------------------------------------
# Minimal obs object compatible with the interpreter
# ---------------------------------------------------------------------------
class Obs:
    def __init__(self, planets, initial_planets=None, fleets=None,
                 next_fleet_id=100, comets=None, comet_planet_ids=None,
                 angular_velocity=0.0):
        self.planets          = [list(p) for p in planets]
        self.initial_planets  = [list(p) for p in (initial_planets if initial_planets is not None else planets)]
        self.fleets           = [list(f) for f in (fleets or [])]
        self.next_fleet_id    = next_fleet_id
        self.comets           = comets or []
        self.comet_planet_ids = comet_planet_ids or []
        self.angular_velocity = angular_velocity


# Owner -> colour
_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}


def simulate(obs, n_steps):
    """Step the interpreter n times; return a snapshot list."""
    snapshots = []
    for step in range(n_steps):
        snapshots.append({
            'step':    step,
            'planets': [p[:] for p in obs.planets],
            'fleets':  [f[:] for f in obs.fleets],
        })
        interpreter(obs, [[], []], step)
    return snapshots


def make_animation(snapshots, title='', interval=150):
    """Animate a snapshot list produced by simulate()."""
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')

    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100)
        ax.set_ylim(100, 0)   # y decreases downward: 100 at top, 0 at bottom
        ax.set_aspect('equal')
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)

        # Sun
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))

        # Planets
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y, str(ships),
                    ha='center', va='center', color='white',
                    fontsize=7, fontweight='bold', zorder=4)

        # Fleets
        for f in snap['fleets']:
            fid, owner, x, y, angle, from_id, ships = f
            c = _COLORS.get(owner, '#888888')
            ax.plot(x, y, 'D', color=c, markersize=5, zorder=5)
            ax.text(x + 1.5, y + 1.5, str(ships), color=c, fontsize=5, zorder=6)

        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

## Test 1 — Planet production
One static planet with `production=3`. Ships should increase by 3 every step.

In [3]:
# planet: id=0, owner=0, x=10, y=10, radius=5, ships=1, production=3
obs1 = Obs(
    planets=[[0, 0, 10.0, 10.0, 5.0, 1, 3]],
    angular_velocity=0.0,
)
snaps1 = simulate(obs1, 10)
make_animation(snaps1, title='Test 1 — Planet Production (prod=3)', interval=400)

## Test 2 — Fleet near planet
Planet at (10, 10) with `production=0`. Fleet at (10, 90) with `angle=0` (east) and 50 ships.
The fleet moves east and eventually leaves the board.

In [4]:
# planet: id=0, owner=0, x=10, y=10, radius=5, ships=1, production=0
# fleet:  id=0, owner=0, x=10, y=90, angle=0 (east), from_planet=0, ships=50
obs2 = Obs(
    planets=[[0, 0, 10.0, 10.0, 5.0, 1, 0]],
    fleets=[[0, 0, 10.0, 90.0, 3 * math.pi / 2, 0, 50]],
    next_fleet_id=1,
    angular_velocity=0.0,
)
snaps2 = simulate(obs2, 30)
make_animation(snaps2, title='Test 2 — Fleet (angle=0) near Planet at (10,10)', interval=120)

## Test 3 — Planet rotation (period = 8 steps)
Planet at (50, 30), orbital radius = 20.  
`angular_velocity = π/4` → full orbit in `2π / (π/4) = 8` steps.

In [5]:
# planet: id=0, owner=0, x=50, y=30, radius=5, ships=1, production=1
# angular_velocity = pi/4  →  period = 8 steps
obs3 = Obs(
    planets=[[0, 0, 50.0, 30.0, 5.0, 1, 1]],
    angular_velocity=math.pi / 4,
)
snaps3 = simulate(obs3, 9)   # 0..8 covers one full orbit
make_animation(snaps3, title='Test 3 — Planet Rotation  ω=π/4  (period=8 steps)', interval=300)

## Test 4 — Fleet toward the sun
Planet at (50, 30). Fleet launched with `angle=π` (west).  
The fleet travels west from (50, 30) and exits the board — it does **not** reach the sun.

> To send a fleet into the sun from (50, 30), the required angle is `π/2` (south).

In [6]:
# planet: id=0, owner=0, x=50, y=30, radius=5, ships=0, production=0
# fleet:  id=0, owner=0, x=50, y=30, angle=pi (west), from_planet=0, ships=50
obs4 = Obs(
    planets=[[0, 0, 50.0, 70.0, 5.0, 0, 0]],
    fleets=[[0, 0, 50.0, 30.0, 1 * math.pi / 2, 0, 50]],
    next_fleet_id=1,
    angular_velocity=0.0,
)
snaps4 = simulate(obs4, 20)
make_animation(snaps4, title='Test 4 — Fleet angle=π (west) from (50,30)', interval=150)

# ── corrected version: angle=π/2 actually reaches the sun ──────────────────
# obs4_sun = Obs(
#     planets=[[0, 0, 50.0, 30.0, 5.0, 0, 0]],
#     fleets=[[0, 0, 50.0, 30.0, math.pi/2, 0, 50]],
#     next_fleet_id=1,
# )
# snaps4_sun = simulate(obs4_sun, 15)
# make_animation(snaps4_sun, title='Test 4 — Fleet angle=π/2 → hits the sun')

## Test 5 — Kaggle env parity
Two random agents play against each other. For every step the **local `interpreter`** is seeded from the current Kaggle env state with the same actions. Both results are compared and animated side-by-side. Red titles flag diverging steps.

In [7]:
import kaggle_environments as ke
import random, math, copy

# ── Random agent ────────────────────────────────────────────────────────────
def random_agent_fn(obs):
    """Send half the ships from a random owned planet in a random direction."""
    player = obs.player
    my_planets = [p for p in obs.planets if p[1] == player]
    if not my_planets:
        return []
    planet = random.choice(my_planets)
    ships = planet[5] // 2
    if ships < 1:
        return []
    return [[planet[0], random.uniform(0, 2 * math.pi), ships]]


# ── Per-step diff ───────────────────────────────────────────────────────────
def compare_planets(k_planets, l_planets):
    msgs = []
    kmap = {p[0]: p for p in k_planets}
    lmap = {p[0]: p for p in l_planets}
    fields = ['id', 'owner', 'x', 'y', 'radius', 'ships', 'production']
    for pid in sorted(set(kmap) | set(lmap)):
        kp, lp = kmap.get(pid), lmap.get(pid)
        if kp is None:
            msgs.append(f"  planet {pid}: missing in kaggle")
        elif lp is None:
            msgs.append(f"  planet {pid}: missing in local")
        else:
            for i, name in enumerate(fields):
                kv, lv = float(kp[i]), float(lp[i])
                if not math.isclose(kv, lv, rel_tol=1e-4, abs_tol=1e-4):
                    msgs.append(f"  planet {pid}.{name}: kaggle={kv:.6g}  local={lv:.6g}")
    return msgs


# ── Side-by-side animation ──────────────────────────────────────────────────
def make_dual_animation(k_snaps, l_snaps, diff_steps=(), title='', interval=200):
    diff_set = set(diff_steps)
    n = min(len(k_snaps), len(l_snaps))
    fig, (ax_k, ax_l) = plt.subplots(1, 2, figsize=(12, 6))
    fig.patch.set_facecolor('#111122')
    fig.suptitle(title, color='white', fontsize=12)

    def draw_panel(ax, snap, label):
        ax.cla()
        ax.set_xlim(0, 100)
        ax.set_ylim(100, 0)
        ax.set_aspect('equal')
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444444')
        s = snap['step']
        tc = '#ff4444' if s in diff_set else 'white'
        ax.set_title(f"{label}  step {s}", color=tc, fontsize=10)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y, str(ships), ha='center', va='center',
                    color='white', fontsize=7, fontweight='bold', zorder=4)
        for f in snap['fleets']:
            fid, owner, x, y, angle, from_id, ships = f
            c = _COLORS.get(owner, '#888888')
            ax.plot(x, y, 'D', color=c, markersize=5, zorder=5)

    def draw(frame):
        draw_panel(ax_k, k_snaps[frame], 'Kaggle env')
        draw_panel(ax_l, l_snaps[frame], 'Local interpreter')
        return []

    ani = animation.FuncAnimation(fig, draw, frames=n, interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())


# ── Run parity test ──────────────────────────────────────────────────────────
N_STEPS = 60
SEED    = 42
random.seed(SEED)

env = ke.make("orbit_wars", debug=False)
env.reset(2)

k_snaps, l_snaps, diffs = [], [], []

# Step 0: initial state (identical for both)
init_obs = env.state[0].observation
init_snap = {
    'step':    0,
    'planets': [list(p) for p in init_obs.planets],
    'fleets':  [list(f) for f in init_obs.fleets],
}
k_snaps.append(init_snap)
l_snaps.append(copy.deepcopy(init_snap))

for step in range(N_STEPS):
    obs_k0 = env.state[0].observation
    obs_k1 = env.state[1].observation

    # Same random actions for both environments
    action0 = random_agent_fn(obs_k0)
    action1 = random_agent_fn(obs_k1)

    # Local interpreter seeded from current Kaggle state
    local_obs = Obs(
        planets          = [list(p) for p in obs_k0.planets],
        initial_planets  = [list(p) for p in obs_k0.initial_planets],
        fleets           = [list(f) for f in obs_k0.fleets],
        next_fleet_id    = obs_k0.next_fleet_id,
        comets           = copy.deepcopy(obs_k0.comets),
        comet_planet_ids = list(obs_k0.comet_planet_ids),
        angular_velocity = obs_k0.angular_velocity,
    )
    interpreter(local_obs, [action0, action1], step)

    # Advance Kaggle env with the same actions
    env.step([action0, action1])
    new_obs_k0 = env.state[0].observation

    # Capture after-step snapshots
    k_snaps.append({
        'step':    step + 1,
        'planets': [list(p) for p in new_obs_k0.planets],
        'fleets':  [list(f) for f in new_obs_k0.fleets],
    })
    l_snaps.append({
        'step':    step + 1,
        'planets': [list(p) for p in local_obs.planets],
        'fleets':  [list(f) for f in local_obs.fleets],
    })

    # Diff
    msgs = compare_planets(new_obs_k0.planets, local_obs.planets)
    if msgs:
        diffs.append((step + 1, msgs))

# ── Report ───────────────────────────────────────────────────────────────────
if diffs:
    n_issues = sum(len(m) for _, m in diffs)
    print(f"FAIL — {n_issues} difference(s) across {len(diffs)} step(s):")
    for s, msgs in diffs[:5]:
        print(f"\nAfter step {s}:")
        for m in msgs:
            print(m)
else:
    print(f"PASS — all {N_STEPS} steps match")

diff_steps = {s for s, _ in diffs}
make_dual_animation(
    k_snaps, l_snaps, diff_steps,
    title='Test 5 — Kaggle env vs Local interpreter',
    interval=200,
)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 17.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_goofspiel
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hearts
[kaggle_environments.envs.open_sp